# Notebook Complementario: Experimentos Secundarios y Resultados Adicionales

**Máster en Ciencia de Datos — Big Data Analytics**  
**Proyecto:** Detección de Comunidades en Twitter con Louvain + Spark

Este notebook contiene experimentos adicionales, análisis exploratorios complementarios y resultados secundarios que apoyan el notebook principal.

---

In [1]:
# ORIGINAL: Configuración del entorno para el notebook complementario
import os, sys, time, warnings, json, gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import community as community_louvain
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 110

DATA_DIR   = Path('../data')
JSONL_PATH = DATA_DIR / 'tweets.jsonl'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName('LouvainTweets_Complementario')
    .master('local[*]')
    .config('spark.driver.memory', '6g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.ui.enabled', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
sc = spark.sparkContext

print(f'Spark {spark.version} | {sc.defaultParallelism} cores | Python {sys.version.split()[0]}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 19:40:44 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.1.160 instead (on interface enx00e04c68022a)
26/04/18 19:40:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 19:40:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 | 18 cores | Python 3.11.9


In [2]:
# ORIGINAL: Carga del dataset
raw_df = spark.read.json(str(JSONL_PATH))

tweets_df = raw_df.select(
    F.col('id_str').alias('tweet_id'),
    F.col('text'),
    F.col('created_at'),
    F.col('retweet_count'),
    F.col('in_reply_to_user_id_str').alias('reply_to_user_id'),
    F.col('user.id_str').alias('user_id'),
    F.col('user.screen_name').alias('screen_name'),
    F.col('user.followers_count').alias('followers'),
    F.col('user.friends_count').alias('following'),
    F.col('user.statuses_count').alias('tweet_count'),
    F.col('user.lang').alias('lang'),
    F.col('user.verified').alias('verified'),
    F.col('entities.user_mentions').alias('mentions'),
    F.col('entities.hashtags').alias('hashtags'),
    F.col('text').startswith('RT @').alias('is_retweet'),
    F.regexp_extract(F.col('text'), r'^RT @([^:]+):', 1).alias('rt_source_name'),
    F.col('source'),
).cache()

total_tweets = tweets_df.count()
print(f'Dataset cargado: {total_tweets:,} tweets')

Dataset cargado: 1,000,000 tweets


## A. Análisis Exploratorio de Datos Extendido

In [3]:
# ORIGINAL: Análisis de la distribución de grado de usuarios (ley de potencias)
# Motivación: confirmar que la red tiene estructura libre de escala

# Tabla de lookup screen_name → user_id
user_lookup = (
    tweets_df
    .select(F.lower(F.col('screen_name')).alias('sn'), F.col('user_id'))
    .dropDuplicates(['sn'])
)

# Reconstruir aristas
edges_rt = (
    tweets_df.filter(F.col('is_retweet'))
    .select(F.col('user_id').alias('src'), F.lower(F.col('rt_source_name')).alias('sn'))
    .filter(F.col('sn') != '')
    .join(user_lookup, on='sn', how='inner')
    .select('src', F.col('user_id').alias('dst'), F.lit(2).alias('weight'))
    .filter(F.col('src') != F.col('dst'))
)
edges_mentions = (
    tweets_df
    .select(F.col('user_id').alias('src'), F.explode('mentions').alias('m'))
    .select('src', F.col('m.id_str').alias('dst'), F.lit(1).alias('weight'))
    .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
    .filter(F.col('src') != F.col('dst'))
)
edges_reply = (
    tweets_df.filter(F.col('reply_to_user_id').isNotNull())
    .select(F.col('user_id').alias('src'), F.col('reply_to_user_id').alias('dst'), F.lit(1).alias('weight'))
    .filter(F.col('src') != F.col('dst'))
)
edges_df = (
    edges_rt.union(edges_mentions).union(edges_reply)
    .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
    .groupBy('src', 'dst').agg(F.sum('weight').alias('weight'))
    .cache()
)
n_edges = edges_df.count()

# Distribución de grado (out-degree ponderado)
degree_dist = (
    edges_df.groupBy('src').agg(F.sum('weight').alias('out_degree'))
    .toPandas()
)

print(f'Nodos con al menos 1 arista saliente: {len(degree_dist):,}')
print(f'Grado máximo: {degree_dist["out_degree"].max():.0f}')
print(f'Grado medio: {degree_dist["out_degree"].mean():.2f}')
print(f'Grado mediana: {degree_dist["out_degree"].median():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución normal
axes[0].hist(degree_dist['out_degree'].clip(upper=200), bins=80,
             color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Distribución de grado ponderado (out)', fontweight='bold')
axes[0].set_xlabel('Grado ponderado')
axes[0].set_ylabel('Frecuencia')
axes[0].set_yscale('log')

# Log-log para verificar ley de potencias
deg_counts = degree_dist['out_degree'].value_counts().sort_index()
axes[1].loglog(deg_counts.index, deg_counts.values, '.', markersize=3, color='coral')
axes[1].set_title('Distribución de grado (escala log-log)', fontweight='bold')
axes[1].set_xlabel('Grado ponderado (log)')
axes[1].set_ylabel('Frecuencia (log)')
axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle('Estructura de Red — Propiedad de Ley de Potencias', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/degree_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: data/degree_distribution.png')

Nodos con al menos 1 arista saliente: 391,901
Grado máximo: 1272
Grado medio: 3.99
Grado mediana: 3


Guardado: data/degree_distribution.png


## B. Comparativa: Louvain Spark vs. Louvain Secuencial

In [4]:
# ORIGINAL: Comparativa entre Louvain secuencial (python-louvain) y paralelizado (Spark)
# Objetivo: verificar que la paralelización no degrada la calidad del resultado

# Usar subconjunto pequeño para que ambos sean comparables en tiempo
edges_pd_full = edges_df.toPandas()

MAX_NODES_COMP = 10_000
degree_small = (
    edges_df
    .select(F.col('src').alias('node'), F.col('weight'))
    .union(edges_df.select(F.col('dst').alias('node'), F.col('weight')))
    .groupBy('node').agg(F.sum('weight').alias('degree'))
    .orderBy(F.desc('degree')).limit(MAX_NODES_COMP)
)
top_nodes_small = set(degree_small.select('node').toPandas()['node'].tolist())
edges_small = edges_pd_full[
    edges_pd_full['src'].isin(top_nodes_small) & edges_pd_full['dst'].isin(top_nodes_small)
].reset_index(drop=True)

print(f'Subgrafo de comparativa: {len(top_nodes_small):,} nodos, {len(edges_small):,} aristas')

# Construir grafo NetworkX
G_comp = nx.Graph()
for _, row in edges_small.iterrows():
    u, v, w = str(row['src']), str(row['dst']), float(row['weight'])
    if G_comp.has_edge(u, v):
        G_comp[u][v]['weight'] += w
    else:
        G_comp.add_edge(u, v, weight=w)

# ── Louvain SECUENCIAL (python-louvain) ────────────────────────────────────────
# Fuente: https://python-louvain.readthedocs.io/
# Licencia: BSD. Adaptado: solo llamada directa a best_partition
t0 = time.time()
part_seq = community_louvain.best_partition(G_comp, weight='weight')
t_seq = time.time() - t0
Q_seq = community_louvain.modularity(part_seq, G_comp, weight='weight')
n_comm_seq = len(set(part_seq.values()))

print(f'\nLouvain SECUENCIAL:')
print(f'  Tiempo     : {t_seq:.2f}s')
print(f'  Q          : {Q_seq:.4f}')
print(f'  Comunidades: {n_comm_seq:,}')

# ── Louvain SPARK (3 iteraciones sobre subgrafo) ──────────────────────────────
def run_louvain_spark_small(edges_pd, sc, max_iter=5):
    """Versión compacta ORIGINAL para comparativa."""
    G = nx.Graph()
    for _, row in edges_pd.iterrows():
        u, v, w = str(row['src']), str(row['dst']), float(row['weight'])
        if G.has_edge(u, v): G[u][v]['weight'] += w
        else: G.add_edge(u, v, weight=w)
    partition = {n: i for i, n in enumerate(G.nodes())}
    m = G.size(weight='weight')
    best_Q, best_part = -1, dict(partition)

    for _ in range(max_iter):
        adj_bc  = sc.broadcast({u: dict(G[u]) for u in G.nodes()})
        part_bc = sc.broadcast(dict(partition))
        m_bc    = sc.broadcast(m)

        def bc_best(nc):
            node, cc = nc
            adj, part, tot = adj_bc.value, part_bc.value, m_bc.value
            nbs = {nb: d.get('weight',1) for nb, d in adj.get(node,{}).items()}
            if not nbs: return (node, cc)
            cw = {}
            for nb, w in nbs.items():
                c = part.get(nb,-1); cw[c] = cw.get(c,0)+w
            ki = sum(nbs.values()); bc, bdQ = cc, 0.0
            for comm, kin in cw.items():
                if comm==cc: continue
                sig = sum(sum(d.get('weight',1) for d in adj.get(u,{}).values())
                          for u, c in part.items() if c==comm)
                dq = (kin/tot) - (sig*ki)/(2*tot**2)
                if dq>bdQ: bdQ=dq; bc=comm
            return (node, bc)

        rdd = sc.parallelize(list(partition.items()), sc.defaultParallelism)
        changed_map = dict(rdd.map(bc_best).collect())
        changed = any(changed_map[n]!=partition[n] for n in partition)
        partition = changed_map
        adj_bc.unpersist(); part_bc.unpersist()
        Q = community_louvain.modularity(partition, G, weight='weight')
        if Q > best_Q: best_Q = Q; best_part = dict(partition)
        if not changed: break
    return best_part, best_Q

t0 = time.time()
part_spark, Q_spark = run_louvain_spark_small(edges_small, sc, max_iter=5)
t_spark = time.time() - t0
n_comm_spark = len(set(part_spark.values()))

print(f'\nLouvain SPARK (paralelizado):')
print(f'  Tiempo     : {t_spark:.2f}s')
print(f'  Q          : {Q_spark:.4f}')
print(f'  Comunidades: {n_comm_spark:,}')

print(f'\nComparativa:')
print(f'  Diferencia Q         : {abs(Q_seq - Q_spark):.4f}')
print(f'  Diferencia comunidades: {abs(n_comm_seq - n_comm_spark)}')
print(f'  Overhead Spark       : {t_spark/t_seq:.1f}x (esperado > 1 en local)')

Subgrafo de comparativa: 10,000 nodos, 82,955 aristas



Louvain SECUENCIAL:
  Tiempo     : 2.55s
  Q          : 0.5081
  Comunidades: 214



Louvain SPARK (paralelizado):
  Tiempo     : 31.77s
  Q          : 0.0173
  Comunidades: 2,901

Comparativa:
  Diferencia Q         : 0.4908
  Diferencia comunidades: 2687
  Overhead Spark       : 12.5x (esperado > 1 en local)


In [5]:
# ORIGINAL: Visualización comparativa Louvain secuencial vs Spark

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

methods = ['Secuencial\n(python-louvain)', 'Spark\n(paralelizado)']
qs = [Q_seq, Q_spark]
times = [t_seq, t_spark]
n_comms = [n_comm_seq, n_comm_spark]
colors = ['steelblue', 'coral']

axes[0].bar(methods, qs, color=colors, edgecolor='white')
axes[0].set_title('Modularidad Q', fontweight='bold')
axes[0].set_ylabel('Q')
axes[0].set_ylim(0, max(qs) * 1.3)
for i, q in enumerate(qs):
    axes[0].text(i, q + 0.002, f'{q:.4f}', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3)

ax2 = axes[1]
x = range(len(methods))
bars = ax2.bar(x, times, color=colors, edgecolor='white', label='Tiempo (s)')
ax3 = ax2.twinx()
ax3.plot(x, n_comms, 'D--', color='#2C3E50', markersize=10, linewidth=2, label='Nº comunidades')
ax2.set_title('Tiempo de ejecución y Nº comunidades', fontweight='bold')
ax2.set_ylabel('Tiempo (s)', color='gray')
ax3.set_ylabel('Nº comunidades', color='#2C3E50')
ax2.set_xticks(x)
ax2.set_xticklabels(methods)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Comparativa: Louvain Secuencial vs. Louvain Spark\n(subgrafo: {len(top_nodes_small):,} usuarios)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/comparativa_louvain.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: data/comparativa_louvain.png')

Guardado: data/comparativa_louvain.png


## C. Análisis de Sensibilidad: Impacto del Número de Iteraciones

In [6]:
# ORIGINAL: Estudio de sensibilidad — cómo varía Q y nº comunidades
# en función del número máximo de iteraciones del algoritmo Louvain.

edges_sample = edges_small  # Usar el subgrafo reducido de la sección anterior

G_sens = nx.Graph()
for _, row in edges_sample.iterrows():
    u, v, w = str(row['src']), str(row['dst']), float(row['weight'])
    if G_sens.has_edge(u, v): G_sens[u][v]['weight'] += w
    else: G_sens.add_edge(u, v, weight=w)

sensitivity_results = []
base_partition = {node: i for i, node in enumerate(G_sens.nodes())}
m_sens = G_sens.size(weight='weight')

for max_it in [1, 2, 3, 5, 8, 12]:
    partition_s = dict(base_partition)
    for _ in range(max_it):
        adj_bc  = sc.broadcast({u: dict(G_sens[u]) for u in G_sens.nodes()})
        part_bc = sc.broadcast(dict(partition_s))
        m_bc    = sc.broadcast(m_sens)

        def best_c_s(nc):
            node, cc = nc
            adj, part, tot = adj_bc.value, part_bc.value, m_bc.value
            nbs = {nb: d.get('weight',1) for nb, d in adj.get(node,{}).items()}
            if not nbs: return (node, cc)
            cw = {}
            for nb, w in nbs.items():
                c = part.get(nb,-1); cw[c] = cw.get(c,0)+w
            ki = sum(nbs.values()); bc, bdQ = cc, 0.0
            for comm, kin in cw.items():
                if comm==cc: continue
                sig = sum(sum(d.get('weight',1) for d in adj.get(u,{}).values())
                          for u, c in part.items() if c==comm)
                dq = (kin/tot) - (sig*ki)/(2*tot**2)
                if dq>bdQ: bdQ=dq; bc=comm
            return (node, bc)

        rdd = sc.parallelize(list(partition_s.items()), sc.defaultParallelism)
        partition_s = dict(rdd.map(best_c_s).collect())
        adj_bc.unpersist(); part_bc.unpersist()

    Q_s  = community_louvain.modularity(partition_s, G_sens, weight='weight')
    nc_s = len(set(partition_s.values()))
    sensitivity_results.append({'max_iter': max_it, 'Q': Q_s, 'n_communities': nc_s})
    print(f'  max_iter={max_it:2d} | Q={Q_s:.4f} | {nc_s:,} comunidades')

df_sens = pd.DataFrame(sensitivity_results)
print('\nAnálisis de sensibilidad completado.')

  max_iter= 1 | Q=0.0090 | 4,563 comunidades


  max_iter= 2 | Q=0.0114 | 3,643 comunidades


  max_iter= 3 | Q=0.0130 | 3,303 comunidades


  max_iter= 5 | Q=0.0173 | 2,901 comunidades


  max_iter= 8 | Q=0.0248 | 2,396 comunidades


  max_iter=12 | Q=0.0169 | 2,041 comunidades

Análisis de sensibilidad completado.


In [7]:
# ORIGINAL: Visualización de sensibilidad al número de iteraciones

fig, ax1 = plt.subplots(figsize=(9, 4))

ax1.plot(df_sens['max_iter'], df_sens['Q'], 'o-',
         color='coral', linewidth=2.5, markersize=8, label='Modularidad Q')
ax1.set_xlabel('Número máximo de iteraciones')
ax1.set_ylabel('Modularidad Q', color='coral')
ax1.tick_params(axis='y', labelcolor='coral')

ax2 = ax1.twinx()
ax2.plot(df_sens['max_iter'], df_sens['n_communities'], 's--',
         color='steelblue', linewidth=2, markersize=7, label='Nº comunidades')
ax2.set_ylabel('Nº comunidades', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')

ax1.set_title('Sensibilidad de Louvain al Número de Iteraciones', fontweight='bold', fontsize=12)
ax1.grid(True, alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('../data/sensibilidad_iteraciones.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: data/sensibilidad_iteraciones.png')

Guardado: data/sensibilidad_iteraciones.png


## D. Análisis de Actividad Temporal del Dataset

In [8]:
# ORIGINAL: Análisis temporal de la actividad en Twitter
# Muestra la distribución de tweets por hora y por tipo

time_pd = tweets_df.select('created_at').toPandas()
time_pd['created_at'] = pd.to_datetime(
    time_pd['created_at'],
    format='%a %b %d %H:%M:%S %z %Y', errors='coerce'
)
time_pd['hour'] = time_pd['created_at'].dt.hour
time_pd = time_pd.dropna(subset=['hour'])
hour_counts = time_pd.groupby('hour').size().reset_index(name='count')

# Distribución por tipo de tweet
n_rt   = tweets_df.filter(F.col('is_retweet')).count()
n_rp   = tweets_df.filter(~F.col('is_retweet') & F.col('reply_to_user_id').isNotNull()).count()
n_orig = tweets_df.filter(~F.col('is_retweet') & F.col('reply_to_user_id').isNull()).count()
tipos  = {'Retweets': n_rt, 'Replies': n_rp, 'Originales': n_orig}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if not hour_counts.empty:
    axes[0].bar(hour_counts['hour'], hour_counts['count'],
                color='mediumpurple', edgecolor='white')
    axes[0].set_title('Actividad por hora del día (UTC)', fontweight='bold')
    axes[0].set_xlabel('Hora')
    axes[0].set_ylabel('Nº tweets')
    axes[0].set_xticks(range(0, 24, 3))
    axes[0].grid(True, alpha=0.3)

axes[1].pie(
    tipos.values(), labels=tipos.keys(),
    colors=['coral', 'steelblue', 'mediumseagreen'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white'}
)
axes[1].set_title('Tipos de tweet', fontweight='bold')

# Top fuentes (cliente Twitter)
import re
source_pd = tweets_df.select('source').toPandas()
source_pd['source_clean'] = source_pd['source'].apply(
    lambda x: re.sub(r'<[^>]+>', '', str(x)).strip() if x else 'Unknown'
)
top_sources = source_pd['source_clean'].value_counts().head(8)
axes[2].barh(top_sources.index[::-1], top_sources.values[::-1],
             color='lightsalmon', edgecolor='white')
axes[2].set_title('Top 8 clientes Twitter', fontweight='bold')
axes[2].set_xlabel('Nº tweets')

plt.suptitle('Análisis Temporal y de Comportamiento — Dataset Twitter',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/temporal_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: data/temporal_analysis.png')

Guardado: data/temporal_analysis.png


## E. Análisis de Usuarios Influyentes por Comunidad

In [9]:
# ORIGINAL: Identificación de usuarios hub (influencers) en el grafo
# Calcula centralidad de grado ponderado y PageRank aproximado

# Calcular grado ponderado de entrada (in-degree) con Spark
indegree_df = (
    edges_df
    .groupBy('dst').agg(F.sum('weight').alias('in_degree'))
    .withColumnRenamed('dst', 'user_id')
)
outdegree_df = (
    edges_df
    .groupBy('src').agg(F.sum('weight').alias('out_degree'))
    .withColumnRenamed('src', 'user_id')
)

# Unir con información de usuario
user_info = (
    tweets_df
    .select(F.col('user_id').cast('string'), 'screen_name', 'followers', 'verified')
    .dropDuplicates(['user_id'])
)

influence_df = (
    user_info
    .join(indegree_df, on='user_id', how='left')
    .join(outdegree_df, on='user_id', how='left')
    .fillna(0, subset=['in_degree', 'out_degree'])
    .withColumn('influence_score',
                F.col('in_degree') * 2 + F.col('followers').cast('double') / 10000)
    .orderBy(F.desc('influence_score'))
    .limit(20)
    .toPandas()
)

print('Top 20 usuarios más influyentes (score = in_degree*2 + followers/10k):')
print('─' * 70)
for _, row in influence_df.iterrows():
    v = '✓' if row.get('verified') else ' '
    print(f"  {v} @{str(row['screen_name']):<25s} | score={float(row['influence_score']):.0f}"
          f" | in={float(row['in_degree']):.0f} | followers={int(row.get('followers') or 0):,}")

Top 20 usuarios más influyentes (score = in_degree*2 + followers/10k):
──────────────────────────────────────────────────────────────────────
  ✓ @BarackObama               | score=53615 | in=25830 | followers=19,545,769
  ✓ @billmaher                 | score=22714 | in=11286 | followers=1,424,621
    @2ChainzLyrics             | score=22212 | in=11101 | followers=99,830
    @KattWillliams             | score=17932 | in=8939 | followers=535,834
    @HumorOrTruth              | score=12387 | in=6180 | followers=274,477
    @ppppolls                  | score=11758 | in=5877 | followers=39,745
    @RetweetDares              | score=11292 | in=5639 | followers=136,194
    @FUN                       | score=11147 | in=5517 | followers=1,125,559
  ✓ @cnnbrk                    | score=9681 | in=4405 | followers=8,707,817
    @itsrealTED                | score=8894 | in=4410 | followers=743,105
    @LMAO_TWITPICS             | score=8401 | in=4191 | followers=189,461
    @RCTV_CONTIGO         

In [10]:
# ORIGINAL: Visualización de usuarios influyentes

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top15 = influence_df.head(15)

axes[0].barh(
    [f"@{n}" for n in top15['screen_name']][::-1],
    top15['in_degree'].astype(float).values[::-1],
    color='coral', edgecolor='white'
)
axes[0].set_title('Top 15 usuarios por In-degree ponderado', fontweight='bold')
axes[0].set_xlabel('In-degree ponderado (suma pesos aristas entrantes)')

axes[1].scatter(
    influence_df['in_degree'].astype(float),
    np.log1p(influence_df['followers'].astype(float)),
    c='steelblue', s=60, alpha=0.7, edgecolors='white', linewidth=0.5
)
axes[1].set_xlabel('In-degree ponderado')
axes[1].set_ylabel('log(1 + followers)')
axes[1].set_title('In-degree vs. Seguidores (top 20 usuarios)', fontweight='bold')
for _, row in influence_df.head(5).iterrows():
    axes[1].annotate(
        f"@{str(row['screen_name'])[:12]}",
        (float(row['in_degree']), np.log1p(float(row['followers'] or 0))),
        fontsize=7, textcoords='offset points', xytext=(3, 3)
    )
axes[1].grid(True, alpha=0.3)

plt.suptitle('Análisis de Influencia — Red de Interacciones Twitter',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/influencers.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: data/influencers.png')

Guardado: data/influencers.png


## F. Métricas de Red Globales

In [11]:
# ORIGINAL: Cálculo de métricas globales de la red sobre el subgrafo reducido
# Se usa el subgrafo G_comp (10K nodos) para que el cómputo sea factible en local

print('Calculando métricas de red del subgrafo (10K nodos)...')
print(f'  Nodos              : {G_comp.number_of_nodes():,}')
print(f'  Aristas            : {G_comp.number_of_edges():,}')

# Densidad
density = nx.density(G_comp)
print(f'  Densidad           : {density:.6f}')

# Componentes conexas
n_components = nx.number_connected_components(G_comp)
largest_cc   = max(nx.connected_components(G_comp), key=len)
print(f'  Componentes        : {n_components}')
print(f'  Componente gigante : {len(largest_cc):,} nodos ({100*len(largest_cc)/G_comp.number_of_nodes():.1f}%)')

# Coeficiente de clustering (aproximado sobre muestras)
sample_nodes = list(G_comp.nodes())[:500]
sub_sample   = G_comp.subgraph(sample_nodes)
avg_clustering = nx.average_clustering(sub_sample, weight='weight')
print(f'  Clustering medio   : {avg_clustering:.4f} (muestra 500 nodos)')

# Diámetro aproximado (sobre el LCC hasta 200 nodos)
lcc_sub = G_comp.subgraph(list(largest_cc)[:200])
if nx.is_connected(lcc_sub):
    diam = nx.diameter(lcc_sub)
    print(f'  Diámetro (sub-LCC) : {diam}')
else:
    print('  Diámetro           : N/A (subgrafo desconectado)')

print('\nMétricas de red calculadas.')

Calculando métricas de red del subgrafo (10K nodos)...
  Nodos              : 8,314
  Aristas            : 80,770
  Densidad           : 0.002337
  Componentes        : 155
  Componente gigante : 7,936 nodos (95.5%)
  Clustering medio   : 0.0099 (muestra 500 nodos)
  Diámetro           : N/A (subgrafo desconectado)

Métricas de red calculadas.


## G. Experimento: Impacto del Tipo de Aristas en la Detección

In [12]:
# ORIGINAL: ¿Qué tipo de aristas contribuye más a la calidad de las comunidades?
# Comparamos la modularidad usando solo RT, solo menciones, y la combinación.

def build_graph_from_spark(edges_spark_df, max_nodes=5000):
    """ORIGINAL: Construye grafo NetworkX desde DataFrame Spark."""
    ep = edges_spark_df.toPandas()
    agg = ep.groupby(['src','dst'])['weight'].sum().reset_index()
    # Filtrar a nodos más activos
    deg = (
        pd.concat([agg[['src','weight']].rename(columns={'src':'node'}),
                   agg[['dst','weight']].rename(columns={'dst':'node'})])
        .groupby('node')['weight'].sum().nlargest(max_nodes)
    )
    top = set(deg.index)
    agg = agg[agg['src'].isin(top) & agg['dst'].isin(top)]
    G = nx.Graph()
    for _, row in agg.iterrows():
        u, v, w = str(row['src']), str(row['dst']), float(row['weight'])
        if G.has_edge(u, v): G[u][v]['weight'] += w
        else: G.add_edge(u, v, weight=w)
    return G

configs = [
    ('Solo RT',        edges_rt),
    ('Solo Menciones', edges_mentions),
    ('RT + Menciones', edges_rt.union(edges_mentions).groupBy('src','dst').agg(F.sum('weight').alias('weight'))),
    ('Todas',          edges_df),
]

edge_type_results = []
for name, edf in configs:
    G_et = build_graph_from_spark(edf, max_nodes=5000)
    if G_et.number_of_edges() == 0:
        print(f'  {name}: sin aristas, omitido')
        continue
    part_et = community_louvain.best_partition(G_et, weight='weight')
    Q_et    = community_louvain.modularity(part_et, G_et, weight='weight')
    nc_et   = len(set(part_et.values()))
    edge_type_results.append({'config': name, 'edges': G_et.number_of_edges(),
                              'nodes': G_et.number_of_nodes(), 'Q': Q_et, 'n_communities': nc_et})
    print(f'  {name:<22s}: {G_et.number_of_nodes():,} nodos, {G_et.number_of_edges():,} aristas | Q={Q_et:.4f} | {nc_et} comunidades')

df_edge_types = pd.DataFrame(edge_type_results)

  Solo RT               : 3,774 nodos, 31,801 aristas | Q=0.5047 | 98 comunidades


  Solo Menciones        : 4,145 nodos, 48,545 aristas | Q=0.4884 | 135 comunidades


  RT + Menciones        : 4,093 nodos, 48,894 aristas | Q=0.4890 | 126 comunidades


  Todas                 : 4,125 nodos, 49,140 aristas | Q=0.4755 | 113 comunidades


In [13]:
# ORIGINAL: Visualización del experimento de tipos de aristas

if not df_edge_types.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    x = range(len(df_edge_types))
    colors = ['#3498DB', '#E74C3C', '#F39C12', '#27AE60']

    axes[0].bar(x, df_edge_types['Q'], color=colors[:len(df_edge_types)], edgecolor='white')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(df_edge_types['config'], rotation=15, ha='right')
    axes[0].set_title('Modularidad por tipo de aristas', fontweight='bold')
    axes[0].set_ylabel('Modularidad Q')
    axes[0].grid(True, alpha=0.3)
    for i, row in df_edge_types.iterrows():
        axes[0].text(i, row['Q'] + 0.001, f"{row['Q']:.4f}", ha='center', fontsize=8)

    axes[1].bar(x, df_edge_types['n_communities'], color=colors[:len(df_edge_types)], edgecolor='white')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(df_edge_types['config'], rotation=15, ha='right')
    axes[1].set_title('Nº de comunidades por tipo de aristas', fontweight='bold')
    axes[1].set_ylabel('Nº comunidades')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Impacto del Tipo de Aristas en la Detección de Comunidades',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/edge_type_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Guardado: data/edge_type_analysis.png')

Guardado: data/edge_type_analysis.png


## H. Resumen de Todos los Experimentos Complementarios

In [14]:
# ORIGINAL: Resumen comparativo de todos los experimentos del notebook complementario

print('=' * 65)
print('RESUMEN — EXPERIMENTOS COMPLEMENTARIOS')
print('=' * 65)

print('\n[A] Distribución de grado:')
print(f'    Grado medio : {degree_dist["out_degree"].mean():.2f}')
print(f'    Grado máx   : {degree_dist["out_degree"].max():.0f}')
print(f'    Ley potencias confirmada: SI (cola pesada visible en log-log)')

print('\n[B] Comparativa secuencial vs Spark:')
print(f'    Q secuencial : {Q_seq:.4f}')
print(f'    Q Spark      : {Q_spark:.4f}')
print(f'    Diferencia   : {abs(Q_seq-Q_spark):.4f}')

print('\n[C] Sensibilidad a max_iter:')
q_iter1 = df_sens[df_sens['max_iter']==1]['Q'].values[0]
q_iterm = df_sens['Q'].max()
print(f'    Q con 1 iter : {q_iter1:.4f}')
print(f'    Q máxima     : {q_iterm:.4f}')

print('\n[F] Métricas globales de red:')
print(f'    Densidad                : {density:.6f}')
print(f'    Componentes conexas     : {n_components}')
print(f'    Componente gigante      : {len(largest_cc):,} nodos')
print(f'    Clustering medio        : {avg_clustering:.4f}')

if not df_edge_types.empty:
    best_cfg = df_edge_types.loc[df_edge_types['Q'].idxmax()]
    print('\n[G] Mejor configuración de aristas:')
    print(f'    Configuración : {best_cfg["config"]}')
    print(f'    Q             : {best_cfg["Q"]:.4f}')

print('\n' + '=' * 65)

spark.stop()
print('Spark detenido. Notebook complementario completado.')

RESUMEN — EXPERIMENTOS COMPLEMENTARIOS

[A] Distribución de grado:
    Grado medio : 3.99
    Grado máx   : 1272
    Ley potencias confirmada: SI (cola pesada visible en log-log)

[B] Comparativa secuencial vs Spark:
    Q secuencial : 0.5081
    Q Spark      : 0.0173
    Diferencia   : 0.4908

[C] Sensibilidad a max_iter:
    Q con 1 iter : 0.0090
    Q máxima     : 0.0248

[F] Métricas globales de red:
    Densidad                : 0.002337
    Componentes conexas     : 155
    Componente gigante      : 7,936 nodos
    Clustering medio        : 0.0099

[G] Mejor configuración de aristas:
    Configuración : Solo RT
    Q             : 0.5047



Spark detenido. Notebook complementario completado.
